In [ ]:
import os
import re
from tqdm import tqdm

In [2]:
# news > query > month > data.jsonl
# praw > query > month > data.jsonl
from data_utils import get_sorted_jsonl_fps_from, \
    get_sorted_dates_from_fps, \
    get_sorted_jsonl_files_of_given_dates_from

QUERIES = ["covid", "coronavirus"]
MONTHS = ["2020-01", "2020-02", "2020-03", "2020-04", "2020-05", "2020-06", 
            "2020-07", "2020-08", "2020-09", "2020-10", "2020-11", "2020-12"]

NEWS_PATH = "../raw/news"
NEWS_SUBDIRS = [os.path.join(NEWS_PATH, query, month) for month in MONTHS for query in QUERIES]
# print(NEWS_SUBDIRS)

PRAW_PATH = "../raw/praw"
PRAW_SUBDIRS = [os.path.join(PRAW_PATH, query, month) for month in MONTHS for query in QUERIES]
# print(PRAW_SUBDIRS)

news_fps = []
praw_fps = []

for news_dir, praw_dir in zip(NEWS_SUBDIRS, PRAW_SUBDIRS):
    curr_news_fps = get_sorted_jsonl_fps_from(news_dir)
    news_fps += curr_news_fps
    curr_dates = get_sorted_dates_from_fps(curr_news_fps)
    praw_fps += get_sorted_jsonl_files_of_given_dates_from(praw_dir, curr_dates)

assert(len(news_fps) == len(praw_fps))
print(*zip(news_fps, praw_fps), sep="\n")


('../raw/news/covid/2020-01/2020-01-01.jsonl', '../raw/praw/covid/2020-01/2020-01-01.jsonl')
('../raw/news/covid/2020-01/2020-01-02.jsonl', '../raw/praw/covid/2020-01/2020-01-02.jsonl')
('../raw/news/covid/2020-01/2020-01-03.jsonl', '../raw/praw/covid/2020-01/2020-01-03.jsonl')
('../raw/news/covid/2020-01/2020-01-04.jsonl', '../raw/praw/covid/2020-01/2020-01-04.jsonl')
('../raw/news/covid/2020-01/2020-01-05.jsonl', '../raw/praw/covid/2020-01/2020-01-05.jsonl')
('../raw/news/covid/2020-01/2020-01-06.jsonl', '../raw/praw/covid/2020-01/2020-01-06.jsonl')
('../raw/news/covid/2020-01/2020-01-07.jsonl', '../raw/praw/covid/2020-01/2020-01-07.jsonl')
('../raw/news/covid/2020-01/2020-01-08.jsonl', '../raw/praw/covid/2020-01/2020-01-08.jsonl')
('../raw/news/covid/2020-01/2020-01-09.jsonl', '../raw/praw/covid/2020-01/2020-01-09.jsonl')
('../raw/news/covid/2020-01/2020-01-10.jsonl', '../raw/praw/covid/2020-01/2020-01-10.jsonl')
('../raw/news/covid/2020-01/2020-01-11.jsonl', '../raw/praw/covid/2020

In [3]:
from data_utils import read_jsonl_as_dict_with_subm_ids_as_keys, get_cleaned_text

def get_top_reactions(json_obj, k=5):
    # get top-level most scored k reactions only
    root_id = json_obj["id"]
    top_reactions_ids = [reaction["id"] for reaction in json_obj["comment timeline"] if root_id in reaction["parent_id"]]
    reactions_scores = [(reaction, reaction["score"]) for reaction in json_obj["comment timeline"] \
                    if reaction["id"] in top_reactions_ids and reaction["body"] not in ["[removed]", "[deleted]"]]
    sorted_reactions = [tup[0] for tup in sorted(reactions_scores, key=lambda x: x[1], reverse=True)]
    return [get_cleaned_text(reaction["body"]) for reaction in sorted_reactions[:k]]

submission_id_to_entry_all = dict()
article_title_to_content_all = dict()

def news_title_content_check(news_obj):
    title, content = news_obj["news title"], news_obj["news content"]
    if "Error"==title: # seems to be a newspaper3k feature? (rare)
        print('"Error" title with content:', content)
        return False
    if "JavaScript is not available." in content:
        print('"JavaScript is not available." in content, title:', title)
        return False
    return True

news_title_content_check_failure_count = 0
news_title_content_check_success_count = 0

for news_fp, praw_fp in zip(news_fps, praw_fps):
    # print("\n\n>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
    # print(news_fp)
    news_dict = read_jsonl_as_dict_with_subm_ids_as_keys(news_fp)
    praw_dict = read_jsonl_as_dict_with_subm_ids_as_keys(praw_fp)
    for subm_id in news_dict:
        assert(subm_id in praw_dict)
        if news_title_content_check(news_dict[subm_id]):
            news_title_content_check_success_count += 1
            if subm_id not in submission_id_to_entry_all:
                # print(subm_id)
                top_reactions = get_top_reactions(praw_dict[subm_id])
                if len(top_reactions) >= 1:
                    # print("number of top reactions:", len(top_reactions), "for", subm_id)
                    article_title = news_dict[subm_id]["news title"] \
                        if len(news_dict[subm_id]["news title"]) > len(news_dict[subm_id]["submission title"]) \
                        else news_dict[subm_id]["submission title"]
                    article_content = get_cleaned_text(news_dict[subm_id]["news content"])
                    if article_title not in article_title_to_content_all:
                        article_title_to_content_all[article_title] = article_content
                    else: # recorded but content is different
                        if article_title_to_content_all[article_title] != article_content:
                            # pick the longer one (heuristic based on eyeballing, rarely happens)
                            if len(article_content) > len(article_title_to_content_all[article_title]):
                                article_title_to_content_all[article_title] = article_content
                    submission_id_to_entry_all[subm_id] = {
                        "subreddit": praw_dict[subm_id]["subreddit"],
                        "article_title": article_title,
                        "article_url": praw_dict[subm_id]["url"],
                        "publisher_domain_address": news_dict[subm_id]["domain"],
                        "post_url": "reddit.com"+praw_dict[subm_id]["permalink"],
                        "post_creation_date": re.findall(r"\d{4}-\d{2}-\d{2}", news_fp)[0],
                        "top_reactions": top_reactions,
                    }
                else: # no top reactions
                    print("No top reactions for", subm_id, "in", news_fp)
            else:
                # covid and coronavirus in the same submission
                print("Already processed", subm_id, "in", news_fp)
        else:
            news_title_content_check_failure_count += 1
            print("news_title_content_check failed for", subm_id, "in", news_fp)

print("news_title_content_check_success_count:", news_title_content_check_success_count)
print("news_title_content_check_failure_count:", news_title_content_check_failure_count)

"JavaScript is not available." in content, title: 
news_title_content_check failed for erz0en in ../raw/news/coronavirus/2020-01/2020-01-21.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for et5e8b in ../raw/news/coronavirus/2020-01/2020-01-23.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for et36tl in ../raw/news/coronavirus/2020-01/2020-01-23.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for et16yp in ../raw/news/coronavirus/2020-01/2020-01-23.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for eszddl in ../raw/news/coronavirus/2020-01/2020-01-23.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for esxvye in ../raw/news/coronavirus/2020-01/2020-01-23.jsonl
"JavaScript is not available." in content, title: 
news_title_content_check failed for esx9ig in ../raw/news/coronavirus/202

In [ ]:
from data_utils import get_langs_set, write_json

articles_in_english = set() # set of article titles
for title, content in tqdm(article_title_to_content_all.items()):
    langs = get_langs_set(content[:500])
    if "ENGLISH" in langs:
        articles_in_english.add(title)

submissions_with_english_reactions = set()
for subm_id, entry in tqdm(submission_id_to_entry_all.items()):
    for reaction in entry["top_reactions"]:
        if subm_id not in submissions_with_english_reactions and \
            "ENGLISH" in get_langs_set(reaction):
                submissions_with_english_reactions.add(subm_id)
                continue

print(len(submission_id_to_entry_all))         # 168446 posts
print(len(submissions_with_english_reactions)) # 157228 en_posts
print(len(article_title_to_content_all))       # 142240 articles
print(len(articles_in_english))                # 120584 en_articles

write_json("./content_included/submission_id_to_entry_all.json", submission_id_to_entry_all)
write_json("./content_included/submission_ids_with_english_reactions.json", list(submissions_with_english_reactions))
write_json("./content_included/article_title_to_content_all.json", article_title_to_content_all)
write_json("./content_included/articles_titles_in_english.json", list(articles_in_english))

100%|██████████| 168446/168446 [01:49<00:00, 1533.61it/s]


168446
157220
142240
120584


In [1]:
from data_utils import read_json

submission_id_to_entry_all = read_json("./content_included/submission_id_to_entry_all.json")
submissions_with_english_reactions = set(read_json("./content_included/submission_ids_with_english_reactions.json"))
article_title_to_content_all = read_json("./content_included/article_title_to_content_all.json")
articles_in_english = set(read_json("./content_included/articles_titles_in_english.json"))

print(len(submission_id_to_entry_all))         # 168446 posts
print(len(submissions_with_english_reactions)) # 157228 en_posts
print(len(article_title_to_content_all))       # 142240 articles
print(len(articles_in_english))                # 120584 en_articles


168446
157220
142240
120584


In [6]:
# keyphrases, starting with old data to save time
from data_utils import read_json, get_MultipartiteRank_keyphrases
title_to_keyphrases_collected_by_older_scripts = read_json("./old/data/title_to_keyphrases.json")
print("existing number of artticles with keyphrases:", len(title_to_keyphrases_collected_by_older_scripts))

english_article_titles_to_keyphrases = dict()
for title in tqdm(article_title_to_content_all):
    if title in articles_in_english:
        if title not in title_to_keyphrases_collected_by_older_scripts:
            keyphrases = get_MultipartiteRank_keyphrases(article_title_to_content_all[title])
            english_article_titles_to_keyphrases[title] = [tup[0] for tup in keyphrases]
        else:
            english_article_titles_to_keyphrases[title] = title_to_keyphrases_collected_by_older_scripts[title]

existing number of artticles with keyphrases: 105320


100%|██████████| 142240/142240 [6:08:07<00:00,  6.44it/s]   


In [ ]:
# from data_utils import get_MultipartiteRank_keyphrases

# english_article_titles_to_keyphrases = dict()
# for title in tqdm(article_title_to_content_all):
#     if title in articles_in_english:
#         keyphrases = get_MultipartiteRank_keyphrases(article_title_to_content_all[title])
#         english_article_titles_to_keyphrases[title] = [tup[0] for tup in keyphrases]

In [12]:

print(len(english_article_titles_to_keyphrases)) # 120584
print(len(articles_in_english))                  # 120584 en_articles
for title in list(english_article_titles_to_keyphrases.keys())[:5]:
    print(title, ":", english_article_titles_to_keyphrases[title])
    print('-----------------------------------')

120584
120584
China confirms new coronavirus as it investigates mystery Wuhan illness: New strain from the family that caused Sars is identified in 15 patients; 59 people reported as having been infected by outbreak traced to Wuhan seafood market : ['photo', 'handout', 'new virus', 'chinese scientists']
-----------------------------------
China Reports First Death From New Virus - The coronavirus, which surfaced in the city of Wuhan, has put the region on alert, but there is no evidence that it can spread among humans. : ['health commission', 'patient', 'people', 'new virus', 'death', 'workers', 'commission', 'initial cases', 'china', 'dozens']
-----------------------------------
Wuhan pneumonia: China steps up efforts to control spread of coronavirus : ['photo', 'coronavirus', 'human-to-human transmission', 'clear evidence', 'reuters', 'chinese officials']
-----------------------------------
Japan confirms first case of coronavirus that has infected dozens in China : ['health ministry

In [13]:
from data_utils import write_json
write_json("./content_included/english_article_titles_to_keyphrases.json", english_article_titles_to_keyphrases)
